# 1 · Diagnose and evaluate

**Can a fluent answer use the wrong policy?** A non-defective item was bought 45 days ago. The customer asks for a full refund.

Current rule: full refund through day 30, store credit after day 30. Defective items need support review.


In [ ]:
import importlib
import os, sys, json
from pathlib import Path

# Find the checkout in local Jupyter or a Databricks Git folder.
start = Path(os.environ.get("WORKSHOP_PROJECT_ROOT", Path.cwd())).resolve()
ROOT = next((p for p in (start, *start.parents) if (p / "west_workshop" / "data.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Set WORKSHOP_PROJECT_ROOT to the workshop checkout, then restart this notebook.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
importlib.invalidate_caches()
os.chdir(ROOT)
MODE = os.environ.get("WORKSHOP_MODE", "saved")  # Explicit opt-in before any model calls.


In [ ]:
from IPython.display import display, HTML, Code
from west_workshop.repair_workshop import notebook_session
from west_workshop.mlflow_view import session_header, inspection_card

session = notebook_session(MODE)
display(HTML(session_header(session)))


## Code view 1: one business check

Compare the answer's eligibility label with the expected label. We check the explanation separately.


In [ ]:
import inspect
from west_workshop.runtime import policy_decision_value

display(Code(inspect.getsource(policy_decision_value), language="python"))


## Code view 2: evaluate the application

`data` holds named cases. `predict_fn` runs the traced application. `scorers` checks its answers.
Saved mode reads the original result. Live mode creates new results and reuses completed batches.


In [ ]:
if session.needs("deployed"):
    import mlflow
    with session.evaluation("deployed") as run:
        result = mlflow.genai.evaluate(
            data=run.data, predict_fn=run.predict_fn, scorers=run.scorers,
        )
        session.record_evaluation("deployed", result, run)


In [ ]:
deployed = session.recorded("deployed")
if not deployed["complete"]:
    raise RuntimeError("This evaluation is incomplete. Inspect its preserved failure before retrying in a new directory.")


## Inspect in MLflow: request → source → check results

Open `day_45_opening`. Which source did the assistant receive? Which check caught the wrong answer?

If no matching tracking server is available, expand the recorded case below.


In [ ]:
display(HTML(inspection_card(session, "deployed", ["day_45_opening"])))


Next: **02_compare_and_decide.ipynb**. Test the judge, change the retrieved policy and inspect what improved.
